# Pescando con IA 

In [78]:
# Instalar paquetes necesarios
!pip install pandas numpy scikit-learn matplotlib

  Using cached matplotlib-3.11.2-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (80 kB)
  Using cached contourpy-1.4.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (3.8 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.5.1-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.2 kB)
  Using cached pyparsing-3.3.3-py3-none-any.whl.metadata (5.9 kB)
Using cached matplotlib-3.11.2-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (9.9 MB)
Using cached contourpy-1.4.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (384 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 7.5 MB/s eta 0:00:00 MB/s eta 0:00:01:01
Using cached kiwisolver-1.5.1-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (1.5 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 8.3 MB/s eta 0:00:00 MB/s eta 0:00:01:01
Usi

In [1]:
# Configure dataset 

import pandas as pd 

df = pd.read_csv("./data/dataset.csv")
df.head(5)
df.tail(6)

,id,NumDots,SubdomainLevel,PathLevel,UrlLength,NumDash,NumDashInHostname,AtSymbol,TildeSymbol,NumUnderscore,...,IframeOrFrame,MissingTitle,ImagesOnlyInForm,SubdomainLevelRT,UrlLengthRT,PctExtResourceUrlsRT,AbnormalExtFormActionR,ExtMetaScriptLinkRT,PctExtNullSelfRedirectHyperlinksRT,CLASS_LABEL
9994,9995,2,1,3,58,1,0,0,0,0,...,0,0,0,1,0,1,1,0,1,0
9995,9996,3,1,1,50,0,0,0,0,0,...,0,0,0,1,1,-1,1,0,1,0
9996,9997,2,1,4,59,1,0,0,0,0,...,1,0,0,1,0,0,1,0,1,0
9997,9998,2,1,4,57,0,0,0,0,0,...,0,0,0,1,0,1,1,0,1,0
9998,9999,3,1,1,49,0,0,0,0,0,...,1,0,0,1,1,0,1,0,1,0
9999,10000,3,1,2,52,3,0,0,0,0,...,0,0,0,1,1,1,1,1,1,0


In [2]:
# data
import numpy as np
arr = df.to_numpy()
arr = arr[::,1::] # eliminar la primera columna correspondiente al id

# Preparacion de los datos para el entrenamiento
X = arr[::,0:48] 
y = arr[::,48]

In [3]:
np.unique_counts(y)

UniqueCountsResult(values=array([0., 1.]), counts=array([5000, 5000]))

In [4]:
from sklearn.model_selection import train_test_split

# Partimos los datos para el entrenamiento con un 20% para datos de pruebas
# Usamos Estratificacion para mantener la proporcion de los datos 1/2
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.2, 
                                                    random_state=0,
                                                    stratify=y)

In [5]:
np.unique_counts(y_test) # Comprobamos que son el mismo numero de muestras 
                         # por cada una de las clases

UniqueCountsResult(values=array([0., 1.]), counts=array([1000, 1000]))

In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, f1_score, recall_score
import matplotlib.pyplot as plt

regularizacion = list(np.logspace(-4, 4, 9))

penalties = ['l1', 'l2']
# Estandarizacion de los datos
scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train)
X_test_std = scaler.transform(X_test)

filas = []
for c in regularizacion: 

    model = LogisticRegression(max_iter=1000000, 
                                    tol=1e-6, 
                                    random_state=0,
                                    C=c)

    # Entrenamiento del modelo
    model.fit(X_train_std, y_train)
    predictions = model.predict(X_test_std)

    # metricas 
    filas.append({    
        'c': c,
        'acc' : accuracy_score(y_test, predictions),
        'f1' : f1_score(y_test, predictions),
        'recall' : recall_score(y_test, predictions),
    }) 

pd.DataFrame(filas)


,c,acc,f1,recall
0,0.0001,0.9065,0.904152,0.882
1,0.0010,0.9325,0.932601,0.934
2,0.0100,0.9365,0.937284,0.949
3,0.1000,0.9415,0.942393,0.957
4,1.0000,0.9405,0.941523,0.958
5,10.0000,0.9435,0.944308,0.958
6,100.0000,0.9440,0.944773,0.958
7,1000.0000,0.9425,0.943322,0.957
8,10000.0000,0.9425,0.943378,0.958


In [ ]:
# Grafica de la matrix de confusion
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Class A', 'Class B'])
disp.plot(cmap=plt.cm.Blues)
plt.show()
